# PROCEDURES

 1. Set Up the Environment for Web Scraping
 Step 1: Navigate to your Documents folder (or preferred directory)
mkdir web_scraping_Lab
cd web_scraping_lab

 Step 2: Create a virtual environment
py -m venv venv

 Step 3: Activate the virtual environment
.\venv\Scripts\activate

 Step 4: Install Jupyter and the IPython kernel
pip install notebook ipykernel

 Step 5: Register your virtual environment as a Jupyter kernel
python -m ipykernel install --user --name=web_scraping_env --display-name "Python (Web Scraping)"

2. Install Required Libraries
 Step 1: Install essential libraries for web scraping
pip install requests beautifulsoup4 lxml pandas

import requests, bs4, lxml, pandas
print("All libraries successfully imported!")

In [12]:
# 3. Send an HTTP GET Request
import requests

url = "https://towardsdatascience.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

response = requests.get(url, headers=headers)

print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")

print("HTML Content Preview (First 500 characters):")
print(response.text[:500])
print("---")

--- Initiating Contact ---
Status Code: 200
HTML Content Preview (First 500 characters):
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8" />
	<script src="https://h030.towardsdatascience.com/script.js"></script><!-- Google Tag Manager -->
<script>
	(function (w, d, s, l, i) {
		w[l] = w[l] || [];
		w[l].push({
			'gtm.start': new Date().getTime(),
			event: 'gtm.js'
		});
		var f = d.getElementsByTagName(s)[0],
			j = d.createElement(s),
			dl = l != 'dataLayer' ? '&l=' + l : '';
		j.async = true;
		j.src =
			'https://www.googletagmanager.com/gtm.js?id=' + i + dl;

---


In [13]:
# 4. Parse HTML Content
import requests
from bs4 import BeautifulSoup

url = "https://towardsdatascience.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status() # Crucial for good error handling

html_content = response.text
soup = BeautifulSoup(html_content, 'html.parser')

print("--- Parsing HTML Content ---")
if soup.title:
    print(f"Parsed Page Title: {soup.title.text.strip()}")
else:
    print("Page title not found. HTML parsing might need adjustment.")
print("---")

--- Parsing HTML Content ---
Parsed Page Title: Towards Data Science
---


In [14]:
# 5. Locate and Extract Specific Data Elements
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    article_links_data = []

    # 1. Target the "In the news" section
    in_the_news_div = soup.find('div', id='mp-itn')
    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    # 2. Target the "On this day" section
    on_this_day_div = soup.find('div', id='mp-otd')
    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10 from all sections):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL: {article['url']}")
    else:
        print("\nNo article links found in the targeted sections.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10 from all sections):
- Title: 'A train crash'
  URL: https://en.wikipedia.org/wiki/2026_Bekasi_train_crash
- Title: 'Jakarta'
  URL: https://en.wikipedia.org/wiki/Jakarta
- Title: 'the London Marathon'
  URL: https://en.wikipedia.org/wiki/2026_London_Marathon
- Title: 'Sabastian Sawe'
  URL: https://en.wikipedia.org/wiki/Sabastian_Sawe
- Title: 'Tigst Assefa'
  URL: https://en.wikipedia.org/wiki/Tigst_Assefa
- Title: 'world record times'
  URL: https://en.wikipedia.org/wiki/Marathon_world_record_progression
- Title: 'Azawad Liberation Front'
  URL: https://en.wikipedia.org/wiki/Azawad_Liberation_Front
- Title: 'Jama'at Nusrat al-Islam wal-Muslimin'
  URL: https://en.wikipedia.org/wiki/Jama%27at_Nusrat_al-Islam_wal-Muslimin
- Title: 'a joint offensive'
  URL: https://en.wikipedia.org/wiki/2026_Mali_offensives
- 

In [15]:
# 6. Extract Data from HTML Tables
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://en.wikipedia.org/wiki/List_of_datasets_for_machine_learning_research"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

print("--- PROCEDURE 6: Extract Data from HTML Tables ---")
target_table = soup.find('table', class_='wikitable')

if target_table:
    table_headers = [th.text.strip() for th in target_table.find('tr').find_all('th')]
    print(f"Table Headers (First 5): {table_headers[:5]}")
    
    table_data = []
    for row in target_table.find_all('tr')[1:6]:
        cells = row.find_all(['td', 'th'])
        cell_texts = [cell.text.strip() for cell in cells]
        table_data.append(cell_texts)
        
    print("\nFirst 5 Data Rows (Raw Lists):")
    for i, row in enumerate(table_data):
        print(f"Row {i+1}: {row[:5]}...")
        
    if table_data:
        try:
            df = pd.DataFrame(table_data, columns=table_headers[:len(table_data[0])])
            print("\nDataFrame Representation (Head of first 3 rows):")
            print(df.head(3).to_string())
        except ValueError as e:
            print(f"\nCould not create DataFrame. Column count mismatch: {e}")
else:
    print("Target table with class 'wikitable' not found on the page.")
print("---")

--- PROCEDURE 6: Extract Data from HTML Tables ---
Table Headers (First 5): ['Type', 'Subtypes']

First 5 Data Rows (Raw Lists):
Row 1: ['Specific category', 'Finance, Economics, Commerce, Societal, Health, Academy, Sports, Food, Agriculture, Travel, Geospatial, Political, Consumer, Transport,  Logistics, Environmental, Real-Estate, Legal, Entertainment, Energy, Hospitality']...
Row 2: ['Scope', 'Supranational Union, National, Subnational, Municipality, Urban, Rural']...
Row 3: ['Language', 'Mandarin Chinese, Spanish, English, Arabic, Hindi, Bengali']...
Row 4: ['Type', 'Tabular, Graph, Text, Image, Sound, Video']...
Row 5: ['Usage', 'Training, validating, and testing']...

DataFrame Representation (Head of first 3 rows):
                Type                                                                                                                                                                                                                   Subtypes
0  Specific category  Financ

In [16]:
# 7.Handle Pagination
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---")
start_url = "http://quotes.toscrape.com/"
base_domain = urlparse(start_url).netloc
all_quotes = []
pages_to_collect = 3
current_page_url = start_url
page_counter = 0

while page_counter < pages_to_collect:
    print(f"\nAttempting to fetch page {page_counter + 1}: {current_page_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'lxml')
        
        quote_elements = soup.find_all('div', class_='quote')
        if quote_elements:
            for quote_item in quote_elements:
                text = quote_item.find('span', class_='text').text.strip()
                author = quote_item.find('small', class_='author').text.strip()
                all_quotes.append({'text': text, 'author': author})
                
            print(f"Collected {len(quote_elements)} quotes from this page.")
        else:
            print("No quote elements found on this page. Stopping.")
            break
            
        next_li = soup.find('li', class_='next')
        if next_li:
            next_link_element = next_li.find('a', href=True)
            if next_link_element:
                next_page_relative_url = next_link_element['href']
                current_page_url = urljoin(start_url, next_page_relative_url)
                page_counter += 1
                time.sleep(1)
            else:
                print("Next page link <a> tag not found within 'next' li.")
                break
        else:
            print("No 'next' pagination link found. Assuming last page.")
            break
            
    except requests.exceptions.RequestException as e:
        print(f"ERROR (Network/HTTP): Could not fetch {current_page_url}: {e}")
        break
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {current_page_url}: {e}")
        break

print(f"\n--- Total Quotes Collected Across {page_counter} pages: {len(all_quotes)} ---")
print("First 5 Collected Quotes (Sample):")
for i, quote in enumerate(all_quotes[:5]):
    print(f'- "{quote["text"]}" - {quote["author"]}')
print("--- End of Procedure ---")

--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---

Attempting to fetch page 1: http://quotes.toscrape.com/
Collected 10 quotes from this page.

Attempting to fetch page 2: http://quotes.toscrape.com/page/2/
Collected 10 quotes from this page.

Attempting to fetch page 3: http://quotes.toscrape.com/page/3/
Collected 10 quotes from this page.

--- Total Quotes Collected Across 3 pages: 30 ---
First 5 Collected Quotes (Sample):
- "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" - Albert Einstein
- "“It is our choices, Harry, that show what we truly are, far more than our abilities.”" - J.K. Rowling
- "“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”" - Albert Einstein
- "“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”" - Jane Austen
- "“Imperfection is beauty, madness is geni

In [17]:
# 8. Clean and Standardize Extracted Data
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

print("--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---")
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')
    
    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]
        
        software_idx = headers_text.index('Software') if 'Software' in headers_text else -1
        creator_idx = headers_text.index('Creator') if 'Creator' in headers_text else -1
        initial_release_idx = headers_text.index('Initial release') if 'Initial release' in headers_text else -1
        
        required_indices = [software_idx, creator_idx, initial_release_idx]
        required_names = ['Software', 'Creator', 'Initial release']
        
        if any(idx == -1 for idx in required_indices):
            print("ERROR: One or more required columns are still missing in the table headers based on current revision.")
            print(f"Expected: {required_names}")
            print("Found headers:", headers_text)
            exit()
            
        cleaned_data_records = []
        data_rows = table.find_all('tr')[1:]
        
        for row_num, row in enumerate(data_rows):
            cols = row.find_all('td')
            if len(cols) > max(required_indices):
                software_name = cols[software_idx].text.strip()
                creator_name = cols[creator_idx].text.strip()
                raw_initial_release = cols[initial_release_idx].text.strip()
                
                clean_initial_release = re.sub(r'\[.*?\]', '', raw_initial_release).strip()
                cleaned_data_records.append({
                    'Software': software_name,
                    'Creator': creator_name,
                    'Initial_Release_Date': clean_initial_release
                })
            else:
                print(f"Skipping row {row_num + 1} due to insufficient columns or unexpected structure.")
                
        print("Cleaned and Standardized Data (First 5 records):")
        for record in cleaned_data_records[:5]:
            print(record)
            
        df_cleaned = pd.DataFrame(cleaned_data_records)
        print("\nDataFrame Info after Cleaning:")
        df_cleaned.info()
        print("\nDataFrame Head (cleaned data):")
        print(df_cleaned.head().to_string())
        
    else:
        print("Table with class 'wikitable' not found. Check URL or table class on the page.")
        
except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")
finally:
    print("--- End of Procedure ---")

--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---
Skipping row 31 due to insufficient columns or unexpected structure.
Cleaned and Standardized Data (First 5 records):
{'Software': 'BigDL', 'Creator': 'Jason Dai (Intel)', 'Initial_Release_Date': '2016'}
{'Software': 'Caffe', 'Creator': 'Berkeley Vision and Learning Center', 'Initial_Release_Date': '2013'}
{'Software': 'Chainer', 'Creator': 'Preferred Networks', 'Initial_Release_Date': '2015'}
{'Software': 'Deeplearning4j', 'Creator': 'Skymind engineering team; Deeplearning4j community; originally Adam Gibson', 'Initial_Release_Date': '2014'}
{'Software': 'DeepSpeed', 'Creator': 'Microsoft', 'Initial_Release_Date': '2019'}

DataFrame Info after Cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Software              30 non-null     str  
 1   Creato

In [18]:
# 9. Implement Error Handling and Robustness
import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 9: Implement Error Handling and Robustness ---")

test_scenarios = {
    "valid_article_page": "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article": "http://httpbin.org/status/404",
    "slow_server_sim": "http://httpbin.org/delay/6",
    "invalid_domain_connect": "http://this.domain.does.not.resolve.abc"
}

for scenario_name, test_url in test_scenarios.items():
    print(f"\nTesting Scenario: '{scenario_name}' at {test_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(test_url, headers=headers, timeout=5)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'lxml')
        
        if scenario_name == "valid_article_page":
            main_title_element = soup.find('h1', id='firstHeading')
        else:
            main_title_element = soup.find('h1')
            
        if main_title_element:
            print(f"SUCCESS: Fetched and found title: '{main_title_element.text.strip()[:100]}...'")
        else:
            print("SUCCESS (with caveat): Fetched page, but main title element not found.")
            
    except requests.exceptions.HTTPError as e:
        print(f"ERROR (HTTP): Status Code {e.response.status_code} for {test_url}. Reason: {e.response.reason}")
    except requests.exceptions.ConnectionError as e:
        print(f"ERROR (Connection): Could not establish connection to {test_url}. Details: {e}")
    except requests.exceptions.Timeout as e:
        print(f"ERROR (Timeout): Request to {test_url} timed out after 5 seconds. Details: {e}")
    except requests.exceptions.RequestException as e:
        print(f"ERROR (General Request): An unexpected request error occurred for {test_url}. Details: {e}")
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {test_url}. Details: {e}")
print("---")

--- PROCEDURE 9: Implement Error Handling and Robustness ---

Testing Scenario: 'valid_article_page' at https://en.wikipedia.org/wiki/Artificial_intelligence
SUCCESS: Fetched and found title: 'Artificial intelligence...'

Testing Scenario: 'non_existent_article' at http://httpbin.org/status/404
ERROR (HTTP): Status Code 404 for http://httpbin.org/status/404. Reason: NOT FOUND

Testing Scenario: 'slow_server_sim' at http://httpbin.org/delay/6
ERROR (Timeout): Request to http://httpbin.org/delay/6 timed out after 5 seconds. Details: HTTPConnectionPool(host='httpbin.org', port=80): Read timed out. (read timeout=5)

Testing Scenario: 'invalid_domain_connect' at http://this.domain.does.not.resolve.abc
ERROR (Connection): Could not establish connection to http://this.domain.does.not.resolve.abc. Details: HTTPConnectionPool(host='this.domain.does.not.resolve.abc', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPConnection(host='this.domain.does.not.resolve.abc',

In [19]:
# 10. Store the Extracted Data
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import sqlite3

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    final_data_to_save = []
    table = soup.find('table', class_='wikitable')
    
    if table:
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]
        table_headers = [re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip() for h in table_headers_raw]
        
        data_rows = table.find_all('tr')[1:]
        
        for i, row in enumerate(data_rows[:15]):
            cols = row.find_all('td')
            row_data = {}
            for j, col in enumerate(cols):
                if j < len(table_headers):
                    header_key = table_headers[j]
                    row_data[header_key] = col.text.strip()
            if row_data:
                final_data_to_save.append(row_data)
                
    else:
        print("ERROR: Table with class 'wikitable' not found on the page. Check URL or table class.")
        
    if final_data_to_save:
        df_output = pd.DataFrame(final_data_to_save)
        print("\n--- Data Ready for Archiving (DataFrame Head) ---")
        print(df_output.head().to_string())
        
        # Step 2: Store data as a CSV file
        csv_file_path = 'deep_learning_software_comparison.csv'
        df_output.to_csv(csv_file_path, index=False, encoding='utf-8')
        print(f"\nData successfully saved to: {csv_file_path} (CSV format)")
        
        # Step 3: Store data as a JSON file
        json_file_path = 'deep_learning_software_comparison.json'
        df_output.to_json(json_file_path, orient='records', indent=4)
        print(f"Data successfully saved to: {json_file_path} (JSON format)")
        
        # Step 4: Store data in an SQLite database
        db_file_path = 'deep_learning_software_comparison.db'
        table_name = 'dl_software_comparison'
        
        conn = None
        try:
            conn = sqlite3.connect(db_file_path)
            df_output.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"Data successfully saved to: {db_file_path} (SQLite database, table: '{table_name}')")
        except Exception as db_err:
            print(f"ERROR (SQLite): Could not save data to SQLite: {db_err}")
        finally:
            if conn:
                conn.close()
                
        # Step 5: Verify files
        all_files_exist = os.path.exists(csv_file_path) and os.path.exists(json_file_path) and os.path.exists(db_file_path)
        if all_files_exist:
            print("\nVerification: CSV, JSON, and SQLite files are all present in the directory.")
        else:
            print("\nVerification Failed: One or more files were not found after saving.")
    else:
        print("No data collected to proceed with storage.")
        
except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")


--- PROCEDURE 10: Store the Extracted Data ---

--- Data Ready for Archiving (DataFrame Head) ---
         Software                                                                     Creator Initial_release Software_license Open_source                                         Platform         Written_in                                     Interface OpenMP_support        OpenCL_support CUDA_support ROCm_support Automatic_differentiation Has_pretrained_models Recurrent_nets Convolutional_nets RBM/DBNs Parallel_execution(multi_node) Actively_developed
0           BigDL                                                           Jason Dai (Intel)            2016       Apache 2.0         Yes                                     Apache Spark              Scala                                 Scala, Python                                                No           No                                             Yes            Yes                Yes                                               

# DATA AND OBSERVATION

In [20]:
# 1
import requests

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    print(f"HTTP Status Code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")


HTTP Status Code: 200


In [21]:
# 2
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    print(f"Page Title (from <title> tag): {soup.title.text}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Page Title (from <title> tag): Web scraping - Wikipedia


In [22]:
# 3
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    main_heading = soup.find('h1', id='firstHeading')
    if main_heading:
        print(f"Main Heading: '{main_heading.text.strip()}'")
    else:
        print("Main heading (h1 with id='firstHeading') not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Main Heading: 'Artificial intelligence'


In [23]:
# 4
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    content_div = soup.find('div', id='mw-content-text') # Adjusted typo in variable name
    first_paragraph = None
    if content_div:
        paragraphs = content_div.find_all('p')
        for p in paragraphs:
            text = p.text.strip()
            if text and not p.find_parent('table', class_='infobox'):
                first_paragraph = text
                break
    if first_paragraph:
        print(f"First Paragraph (truncated): '{first_paragraph[:150]}...'")
    else:
        print("First meaningful paragraph not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

First Paragraph (truncated): 'Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learnin...'


In [24]:
# 5
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    first_quote_div = soup.find('div', class_='quote')
    if first_quote_div:
        quote_text_element = first_quote_div.find('span', class_='text')
        author_element = first_quote_div.find('small', class_='author')
        if quote_text_element and author_element:
            quote = quote_text_element.text.strip()
            author = author_element.text.strip()
            print(f"Sample Quote: '{quote}'")
            print(f"Sample Author: {author}")
        else:
            print("No usable quote text or author element found within the first quote div.")
    else:
        print("No quote div found.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

Sample Quote: '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'
Sample Author: Albert Einstein


In [25]:
# 6
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    first_quote_div = soup.find('div', class_='quote')
    if first_quote_div:
        author_element = first_quote_div.find('small', class_='author')
        if author_element:
            author_name = author_element.text.strip()
            print(f"Author of the first quote: '{author_name}'")
        else:
            print("Author element not found within the first quote div (class='author').")
    else:
        print("First quote container (div with class='quote') not found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Author of the first quote: 'Albert Einstein'


In [26]:
# 7
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    all_quote_divs = soup.find_all('div', class_='quote')
    if all_quote_divs:
        print("Authors found on the page:")
        authors_list = []
        for i, quote_div in enumerate(all_quote_divs):
            author_element = quote_div.find('small', class_='author')
            if author_element:
                author_name = author_element.text.strip()
                authors_list.append(author_name)
                print(f"- {author_name}")
        print(f"\nTotal unique authors found: {len(set(authors_list))}")
    else:
        print("No quote containers (divs with class='quote') found on the page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

Authors found on the page:
- Albert Einstein
- J.K. Rowling
- Albert Einstein
- Jane Austen
- Marilyn Monroe
- Albert Einstein
- André Gide
- Thomas A. Edison
- Eleanor Roosevelt
- Steve Martin

Total unique authors found: 8


In [27]:
# 8
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')
    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]
        population_idx = -1
        for i, header in enumerate(headers_text):
            if 'Population' in header:
                population_idx = i
                break
        if population_idx != -1:
            first_data_row = table.find_all('tr')[1]
            cols = first_data_row.find_all('td')
            if len(cols) > population_idx:
                raw_population = cols[population_idx].text.strip()
                print(f"Raw Population String: '{raw_population}'")
            else:
                print("Population column not found in the first data row.")
        else:
            print("Population header not found in table.")
    else:
        print("Table with class 'wikitable' not found on the page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")

Raw Population String: '8,232,000,000'


In [28]:
# 9
import requests

url = "http://httpbin.org/delay/6"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=2)
    response.raise_for_status()
    print("SUCCESS: Page fetched.")
except requests.exceptions.Timeout as e:
    print(f"ERROR (Timeout): Details: {e}")
except requests.exceptions.RequestException as e:
    print(f"ERROR: {e}")

ERROR (Timeout): Details: HTTPConnectionPool(host='httpbin.org', port=80): Read timed out. (read timeout=2)


In [29]:
# 10
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
scraped_data = []

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')

    quote_elements = soup.find_all('div', class_='quote')
    for q in quote_elements:
        text = q.find('span', class_='text').text.strip()
        author = q.find('small', class_='author').text.strip()
        scraped_data.append({"Quote": text, "Author": author})

    df = pd.DataFrame(scraped_data)
    df.to_csv("quotes.csv", index=False)
    print("Saved quotes.csv")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Saved quotes.csv


In [30]:
# 11
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
scraped_data = []

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    all_quote_divs = soup.find_all('div', class_='quote')

    for q in all_quote_divs:
        text = q.find('span', class_='text').text.strip()
        author = q.find('small', class_='author').text.strip()
        scraped_data.append({"Quote": text, "Author": author})

    df = pd.DataFrame(scraped_data)
    df.to_csv("quotes.csv", index=False)
    print("Saved quotes.csv")

    # Verification step
    if os.path.exists("quotes.csv"):
        print("Verification: quotes.csv exists.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Saved quotes.csv
Verification: quotes.csv exists.


In [31]:
# 12
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}
scraped_data = []

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')

    for q in soup.find_all('div', class_='quote'):
        text = q.find('span', class_='text').text.strip()
        author = q.find('small', class_='author').text.strip()
        scraped_data.append({"Quote": text, "Author": author})

    df = pd.DataFrame(scraped_data)
    df.to_json("quotes.json", orient='records', indent=4)
    print("Saved quotes.json successfully.")
except Exception as e:
    print(f"Error: {e}")

Saved quotes.json successfully.


In [32]:
# 13
import sqlite3
import pandas as pd

data = [{"Quote": "Sample text", "Author": "Sample Author"}]
df = pd.DataFrame(data)

try:
    conn = sqlite3.connect("quotes_db.db")
    df.to_sql("quotes_table", conn, if_exists='replace', index=False)
    print("Saved data to quotes_db.db")
    conn.close()
except Exception as e:
    print(f"Error: {e}")

Saved data to quotes_db.db


In [33]:
# 14
import re

raw_str = "Comparison of software [a]"
cleaned_str = re.sub(r'\[.*?\]', '', raw_str).strip()
print(f"Cleaned string: {cleaned_str}")

Cleaned string: Comparison of software


In [34]:
# 15
import requests
from bs4 import BeautifulSoup
import time

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    soup = BeautifulSoup(response.text, 'lxml')

    next_page = soup.find('li', class_='next')
    if next_page:
        print("Next page element found.")
    else:
        print("No next page found.")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Next page element found.


In [35]:
# 16
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_datasets_for_machine_learning_research"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    rows = table.find_all('tr')
    print(f"Number of rows found: {len(rows)}")
except Exception as e:
    print(f"Error: {e}")

Number of rows found: 14


In [36]:
# 17
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}
try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    print(soup.title.string.strip())
except Exception as e:
    print(f"Error: {e}")

Web scraping - Wikipedia


In [37]:
# 18
import requests

url = "http://httpbin.org/status/404"
try:
    response = requests.get(url, timeout=5)
    if response.status_code == 404:
        print(f"Status 404 handled: {response.status_code}")
except requests.exceptions.HTTPError as e:
    print(f"HTTP error: {e}")

Status 404 handled: 404


In [38]:
# 19
import requests

url = "http://httpbin.org/delay/6"
try:
    requests.get(url, timeout=2)
except requests.exceptions.Timeout:
    print("SUCCESS: Timeout occurred as expected.")

SUCCESS: Timeout occurred as expected.


In [39]:
# 20
from bs4 import BeautifulSoup

html = '<div class="thumbnail"><h4 class="title" title="Laptop A">Laptop A</h4><div class="price">$500</div></div>'
soup = BeautifulSoup(html, 'lxml')

item = soup.select_one(".thumbnail")
name = item.select_one(".title")["title"]
price = item.select_one(".price").text

print(f"Name: {name}, Price: {price}")

Name: Laptop A, Price: $500
